In [ ]:
import pandas as pd

dtype_map = {
    "customer_id": "category",  # 중복이 잦거나 고정 길이 문자열
    "article_id": "int32",  # 10자리 숫자 ID는 int32(약 -21억~+21억)로 충분
    "price": "float32",  # 소수점 가격 데이터 (float64 불필요)
    "sales_channel_id": "int8",  # 1 또는 2 같은 작은 숫자는 int8(-128~127)
    "postal_code": "category",  # 우편번호 등 범주형 데이터
}

customers_df = pd.read_csv(
    r"C:\Users\asd\Desktop\Fitstyle_project\raw_data\customers.csv\customers.csv"
)  # type: ignore  # noqa: E501
articles_df = pd.read_csv(
    r"C:\Users\asd\Desktop\Fitstyle_project\raw_data\articles.csv\articles.csv"
)  # noqa: E501
transactions_df = pd.read_csv(
    r"C:\Users\asd\Desktop\Fitstyle_project\raw_data\transactions_train.csv\transactions_train.csv",
    usecols=["t_dat", "customer_id", "article_id", "price", "sales_channel_id"],
    dtype=dtype_map,
    parse_dates=["t_dat"],  # 날짜는 parse_dates로 바로 datetime 변환
)


In [ ]:
# 1. club_member_status 컬럼의 결측치(NaN)와 'LEFT CLUB' 행을 제외하고 필터링
customers_clean_df = customers_df.dropna(subset=["club_member_status"]).copy()
customers_clean_df = customers_clean_df[
    customers_clean_df["club_member_status"] != "LEFT CLUB"
]

# 2. 제거 후 남은 데이터 확인
print(customers_clean_df["club_member_status"].value_counts())
print(f"제거 전 행 수: {len(customers_df)} -> 제거 후 행 수: {len(customers_clean_df)}")

In [ ]:
customers_clean_df.columns

In [ ]:
customers_clean_df[["FN", "Active"]].value_counts(dropna=False)

In [ ]:
customers_clean_df["FN"].isnull().sum()  # FN 컬럼의 결측치 개수 확인

In [ ]:
# 전체 행 수와 유니크 ID 개수가 같은지 확인 (True면 중복 없음)
print(customers_clean_df["customer_id"].nunique() == len(customers_clean_df))

In [ ]:
customers_clean_df["fashion_news_frequency"].value_counts()  # customer_id 중복 확인

In [ ]:
# 교차표로 일치 여부 확인
print(
    pd.crosstab(
        customers_clean_df["fashion_news_frequency"],
        customers_clean_df["FN"],
        dropna=False,
    )
)

In [ ]:
# 가장 깔끔한 전처리 한 줄
customers_clean_df["fashion_news_subscribed"] = (
    customers_clean_df["fashion_news_frequency"]
    .isin(["Regularly", "Monthly"])
    .astype("int8")
)
customers_clean_df.drop(columns=["FN"], inplace=True)


In [ ]:
pd.crosstab(
    customers_clean_df["Active"],
    customers_clean_df["fashion_news_subscribed"],
    dropna=False,
)

In [ ]:
customers_clean_df["age"].info()  # age 컬럼의 결측치 개수 확인

In [ ]:
# 5개 구간(분위수)으로 균등 분할
labels = ["very young", "young", "middle", "older", "senior"]

customers_clean_df["age_quantile_group"] = pd.qcut(
    customers_clean_df["age"],
    q=5,
    labels=labels,
    duplicates="drop",  # 특정 나이에 데이터가 몰려 경계선이 중복될 때 에러 방지
)

# 구간별 인원수 확인 (결측치 포함)
print(customers_clean_df["age_quantile_group"].value_counts(dropna=False).sort_index())

In [ ]:
# -1(결측치)을 제외한 실제 나이 분포 확인
valid_ages = customers_clean_df[customers_clean_df["age"] > 0]["age"]

# 10대 미만, 10대, 20대, ... 90대 구간 확인
print(
    pd.cut(valid_ages, bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    .value_counts()
    .sort_index()
)

In [ ]:
# 70세 이상 고객들의 나이별 빈도수 확인 (내림차순 정렬)
old_ages = (
    customers_clean_df[customers_clean_df["age"] >= 70]["age"]
    .value_counts()
    .sort_index(ascending=False)
)
print(old_ages.head(3))

In [ ]:
# 0보다 큰 정상 나이 중 가장 어린 나이 10개 확인
print(
    customers_clean_df[customers_clean_df["age"] > 0]["age"]
    .value_counts()
    .sort_index()
    .head(3)
)

In [ ]:
# -1로 채워진 원래 결측치 개수 확인
print("원래 결측치(현재 -1) 개수:", (customers_clean_df["age"] == -1).sum())

In [ ]:
# 1. 나이가 -1인 결측치 행 제거
customers_clean_df = customers_clean_df[customers_clean_df["age"] > 0].copy()

# 2. 나이 컬럼을 int8(1바이트 정수)로 변환해 메모리 최적화
customers_clean_df["age"] = customers_clean_df["age"].astype("int8")

# 3. 정리된 데이터 확인
print("남은 고객 수:", len(customers_clean_df))
print(customers_clean_df["age"].dtype)

In [ ]:
print("고유 postal_code 개수:", customers_clean_df["postal_code"].nunique())
print(customers_clean_df["postal_code"].value_counts().head(10))

In [ ]:
import numpy as np
import pandas as pd

TOP_K = 12          # 원 대회 기준 MAP@12
RECENT_WEEKS = 4    # 인기 상품 집계 기간: train 마지막 4주 (week 2~5)
AGE_BINS = [0, 19, 29, 39, 49, 59, 120]
AGE_LABELS = ["16-19", "20s", "30s", "40s", "50s", "60+"]

age = customers_df["age"].replace(-1, np.nan)
age_group = (
    pd.cut(age, bins=AGE_BINS, labels=AGE_LABELS)
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)
age_group_map = pd.Series(age_group.to_numpy(), index=customers_df["customer_id"])

In [ ]:
recent_train = transactions_df.loc[
    transactions_df["week_idx"].between(2, 1 + RECENT_WEEKS),
    ["customer_id", "article_id"],
]
recent_train = recent_train.assign(age_group=recent_train["customer_id"].map(age_group_map))

global_top = recent_train["article_id"].value_counts().head(TOP_K).index

group_top = (
    recent_train.groupby("age_group")["article_id"]
    .value_counts()
    .groupby(level=0)
    .head(TOP_K)
    .reset_index()[["age_group", "article_id"]]
)

overlap = group_top.groupby("age_group")["article_id"].agg(lambda s: s.isin(global_top).sum())
print(f"전체 top-{TOP_K}와 겹치는 상품 수")
print(overlap.to_string())

# 어떤 상품인지 보기 위해 type 붙이기
group_top_named = group_top.merge(
    articles_df[["article_id", "prod_name", "product_type_name"]],
    on="article_id",
    how="left",
)
print(group_top_named.to_string())

In [ ]:
valid = transactions_df.loc[
    transactions_df["week_idx"] == 8, ["customer_id", "article_id"]
].drop_duplicates()
valid = valid.assign(
    age_group=valid["customer_id"].map(age_group_map),
    hit_global=valid["article_id"].isin(global_top),
)
valid = valid.merge(group_top.assign(hit_group=True), on=["age_group", "article_id"], how="left")
valid["hit_group"] = valid["hit_group"].fillna(False).astype(bool)

# 유저별 recall@12 = 추천 12개 중 맞힌 수 / 그 유저가 산 상품 수
user_recall = valid.groupby(["age_group", "customer_id"], observed=True)[
    ["hit_global", "hit_group"]
].mean()

print("전체 평균 recall@12")
print(user_recall.mean().to_string())
print()
print("나이대별 평균 recall@12")
print(user_recall.groupby(level="age_group").mean().to_string())

# customers_df EDA


## MF/ALS/BPR은 구매 기록만 쓰기 때문에 customers 정보는 사용하지 않는다. 그럼에도 보는 이유!
- 쓰임은 두 가지다.
  1. 신규 유저 fallback (test 거래의 7.55%가 구매 기록 없는 유저)
  2. 그룹별 분석 (나이대에 따라 이미지 feature 효과가 다른지)
- 핵심 연구 질문과는 거리가 있어서 이 두 용도에 쓸 수 있는 상태인지만 확인했다.
- 원칙: customers 컬럼 때문에 유저를 삭제하지 않는다. 유저를 지우면 그 사람의 거래도
  학습/평가에서 빠지는데, CF는 이 컬럼들을 쓰지 않으므로 지울 이유가 없다.

---

## 1. customer_id

확인 이유
- 유저 키가 중복되면 transactions와 merge할 때 거래가 중복 집계된다.

결과
- 행 수와 고유 id 수가 같음. 중복 없음.

진행 방향
- 그대로 사용.

---

## 2. FN, fashion_news_frequency

확인 이유
- 둘 다 패션 뉴스 구독과 관련된 컬럼이라 중복 정보인지 확인했다.

결과
- FN은 값이 1.0 하나이고 나머지(약 89만)는 NaN. NaN은 결측이 아니라 "구독 안 함"으로 추론.
  (공식 설명 없음)
- FN 구독 약 47만, 미구독 약 89만. 대부분 구독하지 않는다.
- fashion_news_frequency 기준 구독 수가 FN보다 많다. 두 컬럼이 서로 다른 시점에
  기록됐거나 동기화가 안 된 것으로 보인다.

진행 방향
- 두 컬럼을 fashion_news_subscribed(0/1) 하나로 합친다.
- 추천 모델 성능에 영향이 작은 컬럼이라 불일치 원인은 더 파지 않는다.

---

## 3. Active

확인 이유
- 활동성이 높은 유저와 낮은 유저의 구매 패턴이 다를 수 있어 그룹 분석 후보로 봤다.

결과
- 값은 1.0 하나, 약 46만 명. 나머지 약 91만은 NaN → FN과 같은 방식으로 "비활성"으로 추론.
- 활성 유저일수록 패션 뉴스 구독률이 높다.

진행 방향
- NaN을 0으로 채워 0/1 플래그로 사용.
- FN과 겹치는 정보가 많아서 둘 다 쓸지는 feature 단계에서 정한다.

---

## 4. club_member_status

확인 이유
- 탈퇴 회원 등 특이한 상태가 있는지 확인했다.

결과
- ACTIVE 약 127만, PRE-CREATE 약 9.2만, 탈퇴와 결측은 합쳐서 0.5% 미만.

진행 방향
- 유저는 제거하지 않는다. 탈퇴 회원도 과거 구매 기록은 유효하다.
- 결측은 Unknown으로 채운다.

---

## 5. age

확인 이유
- 나이대별로 선호 상품이 다르면 신규 유저에게 "같은 나이대 인기 상품"을 추천할 수 있다.
- 나이대별 성능 분석의 기준으로도 쓴다.

결과
- 결측 15,861건 (1% 미만). (13,572와 다른 숫자가 나와서 재확인 필요)
- 최소 16세, 최대 99세.
- 20대가 약 51만 명으로 가장 많고, 30~50대는 20만 명대로 비슷하다.
- 70대 이후 나이대마다 자연스럽게 줄어들어서 99세도 이상치가 아닌 정상 값으로 판단.
- 80~90대는 극소수.

진행 방향
- 10년 단위로 구간을 나누고 60대 이상은 하나로 합친다.
  (16-19 / 20대 / 30대 / 40대 / 50대 / 60+)
- 결측 유저는 삭제하지 않고 Unknown 그룹으로 둔다.
  fallback에서는 Unknown 그룹에 전체 인기 상품을 추천한다.

---

## 6. postal_code

확인 이유
- 지역 정보로 쓸 수 있는지 확인했다.

결과
- 해시 처리되어 있어 실제 지역을 알 수 없다.

진행 방향
- 사용하지 않는다.

## 7. 나이대별 인기 상품 (fallback 검증)

확인 이유
- 신규 유저(test 거래의 7.55%)는 구매 기록이 없어서 인기 상품으로 채워야 한다.
  나이대별 인기가 전체 인기보다 잘 맞으면 fallback과 baseline에 쓸 수 있다.

설정
- 인기 집계: train 마지막 4주 (week 2~5), 평가: valid (week 1)
- 나이대: 16-19 / 20s / 30s / 40s / 50s / 60+ / Unknown
- 지표: 유저별 recall@12 평균

결과 (평가 week 1 / week 2)
- 전체: 0.0210 → 0.0242 / 0.0274 → 0.0300. 두 주 모두 나이대별 인기가 10~15% 높다.
- 16-19가 두 주 모두 가장 크게 개선. 전체 top-12와 절반만 겹쳐서 취향 차이가 가장 크다.
- 30대, 40대는 두 주 모두 recall이 가장 낮다. 인기 상품으로 설명되지 않는 구매가 많은 그룹.
- Unknown은 week 1에서 하락, week 2에서 개선으로 방향이 반대.
  유저 수가 적어서 결과가 흔들리는 것으로 판단.

진행 방향
- 신규 유저 fallback은 나이대별 인기를 쓴다.
- Unknown은 결과가 일관되지 않고 공통 특성도 없는 그룹이라 전체 인기를 쓴다.
- baseline에 나이대별 최근 인기 버전을 추가한다.
- 30~40대는 개인화 모델 결과를 볼 때 따로 분석한다.

### 1. Customers (고객 정보 - 1,371,980건)

| 컬럼명 | 번역 및 상세 설명 | 추천 Dtype |
| :--- | :--- | :--- |
| customer_id | 고객 고유 식별 해시값 (Key) | category / str |
| FN | 패션 뉴스(Fashion News) 수신 여부 (1.0 = 수신) | float64 / int8 |
| Active | 활성 회원 여부 (1.0 = 활성 상태) | float64 / int8 |
| club_member_status | 클럽 멤버십 상태 (ACTIVE, PRE-CREATE 등) | category |
| fashion_news_frequency | 패션 뉴스 수신 주기 (Regularly, Monthly, NONE 등) | category |
| age | 고객 나이 | float64 / Int8 |
| postal_code | 고객 거주 지역 우편번호 암호화 해시값 | category / str |

---

### 2. Articles (상품 메타데이터 - 105,542건)

| 컬럼명 | 번역 및 상세 설명 | 추천 Dtype |
| :--- | :--- | :--- |
| article_id | 상품 고유 번호 (Key, 실제 판매 단위) | int32 / str |
| product_code | 기본 제품 코드 (디자인 베이스 코드) | int32 |
| prod_name | 제품명 (예: Strap top, Jade 등) | str |
| product_type_no | 제품 유형 식별 번호 | int16 |
| product_type_name | 제품 유형 이름 (예: T-shirt, Trousers, Dress) | category |
| product_group_name | 상위 제품군 분류 (예: Garment Upper body, Accessories) | category |
| graphical_appearance_no | 그래픽/패턴 식별 번호 | int16 |
| graphical_appearance_name | 그래픽/패턴 이름 (예: Solid, Stripe, Melange) | category |
| colour_group_code | 색상 그룹 코드 | int16 |
| colour_group_name | 색상 그룹 이름 (예: Black, White, Dark Blue) | category |
| perceived_colour_value_id | 인지 색상 명도/톤 코드 | int16 |
| perceived_colour_value_name | 인지 색상 명도/톤 이름 (예: Dark, Light, Medium) | category |
| perceived_colour_master_id | 메인 기본 색상 계열 코드 | int16 |
| perceived_colour_master_name | 메인 기본 색상 계열 이름 (예: Black, Blue, Beige) | category |
| department_no | 부서/라인 식별 번호 | int16 |
| department_name | 부서/라인 이름 (예: Jersey, Knitwear) | category |
| index_code | 카테고리 인덱스 단축 코드 (예: A, B, C...) | category |
| index_name | 카테고리 인덱스 명칭 (예: Ladieswear, Menswear) | category |
| index_group_no | 인덱스 대분류 그룹 번호 | int8 |
| index_group_name | 인덱스 대분류 이름 (예: Ladies, Divided, Baby/Children) | category |
| section_no | 매장 섹션 식별 번호 | int16 |
| section_name | 매장 섹션 이름 (예: Womens Everyday Collection) | category |
| garment_group_no | 의류 그룹 번호 | int16 |
| garment_group_name | 의류 그룹 명칭 (예: Jersey Fancy, Blouses) | category |
| detail_desc | 상품 상세 설명 문구 (자연어 텍스트) | str |

---

### 3. Transactions (구매 거래 내역 - 31,788,324건)

| 컬럼명 | 번역 및 상세 설명 | 추천 Dtype |
| :--- | :--- | :--- |
| t_dat | 구매 거래 발생 일자 (YYYY-MM-DD) | datetime64[ns] |
| customer_id | 구매 고객 ID (customers 테이블과 매핑) | category / str |
| article_id | 구매 상품 ID (articles 테이블과 매핑) | int32 / str |
| price | 구매 가격 (0~1 범위 등으로 정규화된 가격) | float32 |
| sales_channel_id | 판매 채널 (1: 온라인 몰, 2: 오프라인 매장) | int8 |

In [ ]:
articles_df.info()

In [ ]:
articles_df.isnull().sum()

In [ ]:
articles_df.loc[articles_df["colour_group_code"] == -1, "colour_group_name"].unique()

In [ ]:
articles_df.loc[articles_df["colour_group_code"] == -1].head(10)

In [ ]:
articles_df.columns

In [ ]:
articles_df["colour_group_code"].value_counts()  # 고유 상품명 개수 확인

In [ ]:
def scan_numeric_sentinels(
    df: pd.DataFrame,
    columns: list[str],
    sentinels: tuple[int, ...] = (-1,),
) -> pd.DataFrame:
    """숫자형 컬럼에서 결측치 대체용 sentinel 값의 존재 여부를 스캔한다.

    isnull()로는 감지되지 않는, -1이나 0처럼 실제 값으로 채워진
    결측치 후보를 찾아 컬럼별 개수와 비율을 요약한다.

    Args:
        df: 분석 대상 DataFrame.
        columns: 점검할 숫자형 컬럼명 리스트.
        sentinels: 결측 의심 값 목록.

    Returns:
        컬럼별, sentinel 값별 개수/비율을 담은 요약 DataFrame.
    """
    total = len(df)
    records = []

    for col in columns:
        for sentinel in sentinels:
            count = (df[col] == sentinel).sum()
            if count > 0:
                records.append(
                    {
                        "column": col,
                        "sentinel_value": sentinel,
                        "count": count,
                        "ratio_pct": round(count / total * 100, 3),
                    }
                )

    if not records:
        return pd.DataFrame(columns=["column", "sentinel_value", "count", "ratio_pct"])

    return pd.DataFrame(records).sort_values("ratio_pct", ascending=False)

In [ ]:
numeric_cols = articles_df.select_dtypes(include="number").columns.tolist()
scan_numeric_sentinels(articles_df, numeric_cols, sentinels=(-1, 0, -999, 9999))

In [ ]:
same_rows = articles_df.loc[articles_df["colour_group_code"] == -1].index.equals(
    articles_df.loc[articles_df["colour_group_name"] == "Unknown"].index
)
print(same_rows)

In [ ]:
import os

# 이미지 최상위 폴더 경로 지정 (본인 환경에 맞게 수정)
IMAGE_DIR = r"C:\Users\asd\Desktop\Fitstyle_project\raw_data\images"

# 1. article_id를 반드시 10자리 문자열로 포맷팅 (앞자리 0 복원)
articles_df["article_id_str"] = articles_df["article_id"].astype(str).str.zfill(10)

# 2. H&M 표준 이미지 상대 경로 생성 (0108775015 -> 010/0108775015.jpg)
articles_df["image_rel_path"] = articles_df["article_id_str"].apply(
    lambda x: os.path.join(x[:3], f"{x}.jpg")
)

# 3. 절대 경로 생성
articles_df["image_full_path"] = articles_df["image_rel_path"].apply(
    lambda x: os.path.join(IMAGE_DIR, x)
)

# 4. 실제 디스크에 파일이 존재하는지 확인 (시간이 1~2초 소요될 수 있음)
articles_df["image_exists"] = articles_df["image_full_path"].apply(os.path.exists)

# 5. 매핑 결과 요약 출력
total_articles = len(articles_df)
matched_images = articles_df["image_exists"].sum()
print(f"전체 상품 수: {total_articles:,}개")
print(
    f"이미지 매핑 성공: {matched_images:,}개 ({matched_images / total_articles * 100:.2f}%)"
)
print(f"이미지 누락 상품: {total_articles - matched_images:,}개")

In [ ]:
# 1. 이미지가 누락된 데이터만 추출
missing_img_df = articles_df[~articles_df["image_exists"]]

print(f"총 누락 상품 수: {len(missing_img_df)}개")

# 2. 의류 종류(product_type_name)별 누락 개수 및 비중 확인 (상위 15개)
print("\n=== 1. 누락 상품의 의류 종류(product_type_name) Top 15 ===")
missing_type_summary = (
    missing_img_df["product_type_name"]
    .value_counts()
    .head(15)
    .to_frame(name="missing_count")
)
missing_type_summary["total_count"] = articles_df["product_type_name"].value_counts()
missing_type_summary["missing_rate(%)"] = (
    missing_type_summary["missing_count"] / missing_type_summary["total_count"] * 100
).round(2)
print(missing_type_summary)

# 3. 상위 카테고리(product_group_name)별 누락 분포
print("\n=== 2. 누락 상품의 상위 카테고리(product_group_name) 분포 ===")
print(missing_img_df["product_group_name"].value_counts())

# 4. 타깃 라인(index_group_name)별 누락 분포
print("\n=== 3. 누락 상품의 타깃 라인(index_group_name) 분포 ===")
print(missing_img_df["index_group_name"].value_counts())

# 5. 샘플 메타데이터 5개 출력해보기
print("\n=== 4. 누락 상품 샘플 메타데이터 (5개) ===")
display_cols = [
    "article_id_str",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "detail_desc",
]
print(missing_img_df[display_cols].head())

In [ ]:
# 시각적 속성을 가장 잘 나타내는 핵심 컬럼 선별
visual_meta_cols = [
    "product_type_name",  # 의류 종류 (T-shirt, Dress 등)
    "product_group_name",  # 상위 카테고리 (Garment Upper body 등)
    "graphical_appearance_name",  # 패턴/그래픽 (Solid, Stripe 등)
    "colour_group_name",  # 색상 (Black, White, Dark Blue 등)
    "perceived_colour_value_name",  # 톤/명도 (Dark, Light, Dusky 등)
    "index_group_name",  # 타깃 라인 (Ladies, Men, Baby/Children)
    "detail_desc",  # 텍스트 상세 설명 (CLIP 텍스트 인코더 입력용)
]

# 핵심 컬럼의 고유값 개수 및 결측치 점검
print(articles_df[visual_meta_cols].nunique())
print("\n[결측치 개수]")
print(articles_df[visual_meta_cols].isna().sum())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# 이미지가 존재하는 상품 중 하나 샘플링
sample = articles_df[articles_df["image_exists"]].sample(1).iloc[0]

print("=== 샘플 상품 메타데이터 ===")
print(f"ID: {sample['article_id_str']}")
print(f"상품명: {sample['prod_name']}")
print(f"카테고리: {sample['product_type_name']} ({sample['product_group_name']})")
print(f"패턴: {sample['graphical_appearance_name']}")
print(f"색상: {sample['colour_group_name']}")
print(f"설명: {sample['detail_desc']}")

# 이미지 시각화
img = Image.open(sample["image_full_path"])
plt.figure(figsize=(4, 6))
plt.imshow(img)
plt.title(f"{sample['prod_name']}\n({sample['colour_group_name']})")
plt.axis("off")
plt.show()

In [ ]:
import pandas as pd

# 1. product_group_name이 'Unknown'인 상품들만 필터링
unknown_group_df = articles_df[articles_df["product_group_name"] == "Unknown"]

print(f"product_group_name이 'Unknown'인 상품 수: {len(unknown_group_df):,}개")

# 2. Unknown 상품들의 세부 카테고리(product_type_name) 분포 확인
print("\n=== 1. Unknown 상품들의 세부 카테고리(product_type_name) 분포 ===")
print(unknown_group_df["product_type_name"].value_counts())

# 3. Unknown 상품들의 이미지 존재 여부 확인 (앞서 생성한 image_exists 컬럼 기준)
if "image_exists" in unknown_group_df.columns:
    print("\n=== 2. Unknown 상품들의 이미지 존재 여부 ===")
    print(unknown_group_df["image_exists"].value_counts())

# 4. 실제 상품을 파악하기 위해 주요 정보가 담긴 샘플 10개 출력
print("\n=== 3. Unknown 상품 샘플 목록 (10개) ===")
sample_cols = [
    "article_id_str",
    "prod_name",
    "product_type_name",
    "colour_group_name",
    "detail_desc",
]

# 저장된 컬럼명 형태에 맞춰 존재할 때만 출력
available_cols = [col for col in sample_cols if col in unknown_group_df.columns]
display(unknown_group_df[available_cols].head(10))

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image


def get_image_path(article_id: int, image_dir: str) -> str:
    """article_id로부터 H&M 데이터셋 이미지 경로를 생성한다.

    H&M 데이터셋은 article_id를 10자리로 zero-padding한 뒤,
    앞 3자리를 폴더명으로 사용하는 규칙을 따른다.

    Args:
        article_id: 상품 고유 ID.
        image_dir: 이미지 최상위 디렉토리 경로 (예: 'data/raw/images').

    Returns:
        예상되는 이미지 파일의 전체 경로.
    """
    padded_id = str(article_id).zfill(10)
    folder = padded_id[:3]
    return os.path.join(image_dir, folder, f"{padded_id}.jpg")


def get_existing_image_paths(
    df: pd.DataFrame,
    image_dir: str,
    id_col: str = "article_id",
) -> pd.DataFrame:
    """DataFrame의 article_id에 대해 실제 이미지 존재 여부를 확인한다.

    Args:
        df: article_id 컬럼을 포함한 DataFrame.
        image_dir: 이미지 최상위 디렉토리 경로.
        id_col: article_id 컬럼명.

    Returns:
        'image_path'와 'image_exists' 컬럼이 추가된 DataFrame 복사본.
    """
    result = df.copy()
    result["image_path"] = result[id_col].apply(lambda x: get_image_path(x, image_dir))
    result["image_exists"] = result["image_path"].apply(os.path.exists)
    return result


def plot_image_grid(
    df: pd.DataFrame,
    image_dir: str,
    n_samples: int = 16,
    n_cols: int = 4,
    id_col: str = "article_id",
    label_cols: tuple = ("product_type_name", "detail_desc"),
    random_state: int = 42,
) -> None:
    """조건에 맞는 상품 이미지를 그리드로 시각화한다.

    이미지가 존재하는 항목만 샘플링하여 표시하며,
    제목에는 article_id와 지정된 메타데이터 컬럼을 함께 표기한다.

    Args:
        df: 시각화 대상 DataFrame (예: product_group_name == 'Unknown' 필터링 결과).
        image_dir: 이미지 최상위 디렉토리 경로.
        n_samples: 표시할 이미지 개수.
        n_cols: 그리드 열 개수.
        id_col: article_id 컬럼명.
        label_cols: 제목에 표시할 메타데이터 컬럼 목록.
        random_state: 샘플링 재현성을 위한 시드.

    Raises:
        ValueError: 이미지가 존재하는 항목이 하나도 없을 경우.
    """
    checked = get_existing_image_paths(df, image_dir, id_col)
    available = checked[checked["image_exists"]]

    if available.empty:
        raise ValueError(
            "이미지가 존재하는 항목이 없습니다. image_dir 경로 규칙을 다시 확인하세요."
        )

    sample = available.sample(
        n=min(n_samples, len(available)), random_state=random_state
    )

    n_rows = (len(sample) + n_cols - 1) // n_cols
    _, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(row["image_path"])
        ax.imshow(img)
        title = f"id: {row[id_col]}\n"
        title += "\n".join(str(row[col])[:30] for col in label_cols if col in row)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    n_missing = len(checked) - len(available)
    print(
        f"전체 {len(checked)}개 중 이미지 존재 {len(available)}개, 누락 {n_missing}개"
    )

In [ ]:
unknown_articles = articles_df[articles_df["product_group_name"] == "Unknown"]
plot_image_grid(
    df=unknown_articles,
    image_dir="C:\\Users\\asd\\Desktop\\Fitstyle_project\\raw_data\\images",
    n_samples=16,
    n_cols=5,
)

In [ ]:
def analyze_unknown_impact(
    articles: pd.DataFrame,
    transactions: pd.DataFrame,
    group_col: str = "product_group_name",
    unknown_label: str = "Unknown",
    type_col: str = "product_type_name",
    id_col: str = "article_id",
) -> None:
    """Unknown 그룹 제거가 미치는 영향을 정량적으로 분석한다.

    전체 비율, transaction 연관성, product_type_name을 통한
    그룹 복구 가능성을 확인하여 제거 여부 판단 근거를 제공한다.

    Args:
        articles: 상품 메타데이터 DataFrame.
        transactions: 거래 기록 DataFrame.
        group_col: 상품 그룹 컬럼명.
        unknown_label: 미상 그룹을 나타내는 값.
        type_col: 상품 타입 컬럼명 (그룹 복구 시도용).
        id_col: 상품 ID 컬럼명.
    """
    unknown = articles[articles[group_col] == unknown_label]
    total = len(articles)
    ratio = len(unknown) / total * 100

    print(f"[비율] Unknown {len(unknown)}개 / 전체 {total}개 ({ratio:.3f}%)")

    unknown_ids = set(unknown[id_col])
    tx_count = transactions[id_col].isin(unknown_ids).sum()
    tx_ratio = tx_count / len(transactions) * 100
    print(
        f"[거래 연관성] Unknown 상품 관련 거래 {tx_count}건 "
        f"(전체 거래의 {tx_ratio:.4f}%)"
    )

    type_to_group = (
        articles[articles[group_col] != unknown_label]
        .groupby(type_col)[group_col]
        .agg(lambda x: x.mode().iat[0] if not x.mode().empty else None)
    )
    recoverable = unknown[type_col].isin(type_to_group.index).sum()
    print(
        f"[복구 가능성] product_type_name으로 그룹 유추 가능한 항목: "
        f"{recoverable} / {len(unknown)}"
    )

In [ ]:
analyze_unknown_impact(articles_df, transactions_df)

In [ ]:
import pandas as pd


def summarize_categorical_unknowns(
    df: pd.DataFrame,
    columns: list[str],
    unknown_values: tuple[str, ...] = ("Unknown", "unknown", "NONE"),
) -> pd.DataFrame:
    """범주형 컬럼별 Unknown/결측 비율을 한 번에 요약한다.

    각 컬럼에서 지정된 unknown 값과 실제 NaN의 개수, 전체 대비 비율을
    계산하여, 어떤 컬럼을 우선적으로 검토해야 할지 판단할 근거를 제공한다.

    Args:
        df: 분석 대상 DataFrame (예: articles).
        columns: 점검할 범주형 컬럼명 리스트.
        unknown_values: Unknown으로 간주할 문자열 값들.

    Returns:
        컬럼별 unknown_count, nan_count, total, ratio_pct를 담은 요약 DataFrame.
    """
    total = len(df)
    records = []

    for col in columns:
        unknown_count = df[col].isin(unknown_values).sum()
        nan_count = df[col].isna().sum()
        records.append(
            {
                "column": col,
                "unknown_count": unknown_count,
                "nan_count": nan_count,
                "total": total,
                "ratio_pct": round((unknown_count + nan_count) / total * 100, 3),
            }
        )

    summary = pd.DataFrame(records).sort_values("ratio_pct", ascending=False)
    return summary.reset_index(drop=True)

In [ ]:
categorical_cols = [
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "index_name",
    "section_name",
    "garment_group_name",
    "perceived_colour_master_name",
    "department_name",
    "index_group_name",
    "section_name",
]

summarize_categorical_unknowns(articles_df, categorical_cols)

In [ ]:
type_counts = articles_df["product_type_name"].value_counts()

for threshold in [1, 5, 10, 30, 50]:
    low = type_counts[type_counts < threshold]
    print(f"{threshold}개 미만: {len(low)}종, 상품 {low.sum()}개")

print(type_counts[type_counts < 50])

In [ ]:
excluded_groups = [
    "Items",
    "Furniture",
    "Garment and Shoe care",
    "Stationery",
    "Interior textile",
    "Fun",
    "Cosmetic",
    "Unknown",
]
non_fashion_types = [
    "Washing bag",
    "Sewing kit",
    "Stain remover spray",
    "Clothing mist",
    "Wood balls",
    "Zipper head",
    "Marker pen",
    "Side table",
]

fashion_df = articles_df[
    ~articles_df["product_group_name"].isin(excluded_groups)
    & ~articles_df["product_type_name"].isin(non_fashion_types)
]

type_counts = fashion_df["product_type_name"].value_counts()
low = type_counts[type_counts < 50]

print(f"남은 저빈도 type: {len(low)}종, 상품 {low.sum()}개")
print(low.to_string())

In [ ]:
type_merge_mapping = {
    # 표기 불일치
    "Earrings": "Earring",
    "Flat shoes": "Flat shoe",
    "Cap": "Cap/peaked",
    "Sleep Bag": "Sleeping sack",
    # 헤어 액세서리
    "Alice band": "Hair/alice band",
    "Hairband": "Hair/alice band",
    "Headband": "Hair/alice band",
    # 가방류
    "Backpack": "Bag",
    "Cross-body bag": "Bag",
    "Tote bag": "Bag",
    "Shoulder bag": "Bag",
    "Weekend/Gym bag": "Bag",
    "Bumbag": "Bag",
    # 모자류
    "Felt hat": "Hat/beanie",
    "Bucket hat": "Hat/beanie",
    "Straw hat": "Hat/beanie",
}

In [ ]:
non_fashion_types = [
    # 기존
    "Washing bag",
    "Sewing kit",
    "Stain remover spray",
    "Clothing mist",
    "Wood balls",
    "Zipper head",
    "Marker pen",
    "Side table",
    # 저빈도 탐색에서 추가
    "Soft Toys",
    "Waterbottle",
    "Giftbox",
    "Dog Wear",
    "Umbrella",
    "Sleeping sack",
    "Sleep Bag",
    "Baby Bib",
    "Nipple covers",
    "Bra extender",
]

In [ ]:
targets = set(type_merge_mapping.values())
print(targets - set(fashion_df["product_type_name"].unique()))

In [ ]:
excluded = [
    "Items",
    "Furniture",
    "Garment and Shoe care",
    "Stationery",
    "Interior textile",
    "Fun",
    "Cosmetic",
]

garment_unknown = articles_df[articles_df["garment_group_name"] == "Unknown"]
print(f"garment_group_name Unknown 전체: {len(garment_unknown)}건")

overlap = garment_unknown[garment_unknown["product_group_name"].isin(excluded)]
remaining = garment_unknown[~garment_unknown["product_group_name"].isin(excluded)]

print(f"제거 예정 카테고리와 겹침: {len(overlap)}건")
print(
    f"제거 후에도 남는 순수 Unknown: {len(remaining)}건 "
    f"({len(remaining) / len(articles_df) * 100:.3f}%)"
)

print("\nUnknown의 product_group_name 분포:")
print(garment_unknown["product_group_name"].value_counts())

In [ ]:
unique_types = garment_unknown["product_type_name"].value_counts()
print(f"고유 product_type_name 개수: {len(unique_types)}")
print(unique_types)

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image


def get_image_path(article_id: int, image_dir: str) -> str:
    """article_id로부터 H&M 데이터셋 이미지 경로를 생성한다.

    H&M 데이터셋은 article_id를 10자리로 zero-padding한 뒤,
    앞 3자리를 폴더명으로 사용하는 규칙을 따른다.

    Args:
        article_id: 상품 고유 ID.
        image_dir: 이미지 최상위 디렉토리 경로 (예: 'data/raw/images').

    Returns:
        예상되는 이미지 파일의 전체 경로.
    """
    padded_id = str(article_id).zfill(10)
    folder = padded_id[:3]
    return os.path.join(image_dir, folder, f"{padded_id}.jpg")


def get_existing_image_paths(
    df: pd.DataFrame,
    image_dir: str,
    id_col: str = "article_id",
) -> pd.DataFrame:
    """DataFrame의 article_id에 대해 실제 이미지 존재 여부를 확인한다.

    Args:
        df: article_id 컬럼을 포함한 DataFrame.
        image_dir: 이미지 최상위 디렉토리 경로.
        id_col: article_id 컬럼명.

    Returns:
        'image_path'와 'image_exists' 컬럼이 추가된 DataFrame 복사본.
    """
    result = df.copy()
    result["image_path"] = result[id_col].apply(lambda x: get_image_path(x, image_dir))
    result["image_exists"] = result["image_path"].apply(os.path.exists)
    return result


def plot_image_grid(
    df: pd.DataFrame,
    image_dir: str,
    n_samples: int = 16,
    n_cols: int = 4,
    id_col: str = "article_id",
    label_cols: tuple = ("product_type_name", "detail_desc"),
    random_state: int = 42,
) -> None:
    """조건에 맞는 상품 이미지를 그리드로 시각화한다.

    이미지가 존재하는 항목만 샘플링하여 표시하며,
    제목에는 article_id와 지정된 메타데이터 컬럼을 함께 표기한다.

    Args:
        df: 시각화 대상 DataFrame (예: product_group_name == 'Unknown' 필터링 결과).
        image_dir: 이미지 최상위 디렉토리 경로.
        n_samples: 표시할 이미지 개수.
        n_cols: 그리드 열 개수.
        id_col: article_id 컬럼명.
        label_cols: 제목에 표시할 메타데이터 컬럼 목록.
        random_state: 샘플링 재현성을 위한 시드.

    Raises:
        ValueError: 이미지가 존재하는 항목이 하나도 없을 경우.
    """
    checked = get_existing_image_paths(df, image_dir, id_col)
    available = checked[checked["image_exists"]]

    if available.empty:
        raise ValueError(
            "이미지가 존재하는 항목이 없습니다. image_dir 경로 규칙을 다시 확인하세요."
        )

    sample = available.sample(
        n=min(n_samples, len(available)), random_state=random_state
    )

    n_rows = (len(sample) + n_cols - 1) // n_cols
    _, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(row["image_path"])
        ax.imshow(img)
        title = f"id: {row[id_col]}\n"
        title += "\n".join(str(row[col])[:30] for col in label_cols if col in row)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    n_missing = len(checked) - len(available)
    print(
        f"전체 {len(checked)}개 중 이미지 존재 {len(available)}개, 누락 {n_missing}개"
    )

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np


def show_representative_by_type(
    df: pd.DataFrame,
    type_col: str,
    image_dir: str,
    id_col: str = "article_id",
    n_per_type: int = 2,
    n_cols: int = 4,
) -> None:
    """product_type_name별 대표 이미지를 소량씩 확인한다.

    동일 type_col 값을 가진 상품은 garment_group_name이 같을 가능성이
    높다는 전제 하에, 전체를 보는 대신 type별 소수 샘플만 시각화하여
    라벨링 작업량을 줄인다.

    Args:
        df: 대상 DataFrame (garment_group_name == 'Unknown' 필터링 결과).
        type_col: 대표성을 판단할 기준 컬럼 (예: 'product_type_name').
        image_dir: 이미지 최상위 디렉토리 경로.
        id_col: article_id 컬럼명.
        n_per_type: type별로 확인할 이미지 개수.
        n_cols: 그리드 열 개수.
    """
    checked = get_existing_image_paths(df, image_dir, id_col)
    types = checked[type_col].unique()

    for t in types:
        subset = checked[checked[type_col] == t]
        subset = subset[subset["image_exists"]]
        if subset.empty:
            continue
        sample = subset.sample(n=min(n_per_type, len(subset)), random_state=42)

        actual_cols = min(n_cols, len(sample))
        n_rows = (len(sample) + actual_cols - 1) // actual_cols

        fig, axes = plt.subplots(
            n_rows, actual_cols, figsize=(4 * actual_cols, 4 * n_rows)
        )

        # axes를 항상 1차원 리스트/배열로 변환 (단일 Axes 처리 포함)
        axes_flat = np.atleast_1d(axes).flat

        # 이미지 출력
        for ax, (_, row) in zip(axes_flat, sample.iterrows()):
            img = Image.open(row["image_path"])
            ax.imshow(img)
            ax.set_title(f"id: {row[id_col]}", fontsize=8)
            ax.axis("off")

        # 남는 subplot 숨기기
        for ax in list(axes_flat)[len(sample) :]:
            ax.axis("off")

        plt.suptitle(f"product_type_name: {t}  (총 {len(subset)}건)")
        plt.tight_layout()
        plt.show()

In [ ]:
show_representative_by_type(
    garment_unknown,
    type_col="product_type_name",
    image_dir="C:\\Users\\asd\\Desktop\\Fitstyle_project\\raw_data\\images",
)

In [ ]:
print(garment_unknown["product_type_name"].unique().tolist())

In [ ]:
print(
    articles_df.loc[
        articles_df["garment_group_name"] != "Unknown", "garment_group_name"
    ].unique()
)

In [ ]:
manual_garment_mapping = {
    # --- [상의 및 기본 탑] ---
    "T-shirt": "Jersey Basic",
    "Vest top": "Jersey Basic",
    "Top": "Jersey Fancy",
    "Polo shirt": "Jersey Fancy",
    "Shirt": "Shirts",
    "Blouse": "Blouses",
    "Bodysuit": "Jersey Basic",
    "Garment Set": "Dressed",
    # --- [아우터 및 수트류] ---
    "Sweater": "Knitwear",
    "Cardigan": "Knitwear",
    "Hoodie": "Jersey Fancy",
    "Jacket": "Outdoor",
    "Coat": "Outdoor",
    "Outdoor Waistcoat": "Outdoor",
    "Tailored Waistcoat": "Dressed",
    "Blazer": "Dressed",
    # --- [하의 및 드레스/올인원] ---
    "Trousers": "Trousers",
    "Outdoor trousers": "Outdoor",
    "Shorts": "Shorts",
    "Skirt": "Skirts",
    "Dress": "Dresses Ladies",
    "Jumpsuit/Playsuit": "Dressed",
    "Dungarees": "Trousers",
    # --- [속옷, 잠옷 및 수영복] ---
    "Bra": "Under-, Nightwear",
    "Underwear bottom": "Under-, Nightwear",
    "Underwear body": "Under-, Nightwear",
    "Underwear Tights": "Under-, Nightwear",
    "Swimsuit": "Swimwear",
    "Bikini top": "Swimwear",
    "Swimwear top": "Swimwear",
    "Swimwear bottom": "Swimwear",
    "Swimwear set": "Swimwear",
    # --- [신발 및 양말/타이즈] ---
    "Socks": "Socks and Tights",
    "Leggings/Tights": "Socks and Tights",
    "Sneakers": "Shoes",
    "Boots": "Shoes",
    "Bootie": "Shoes",
    "Pumps": "Shoes",
    "Sandals": "Shoes",
    "Heeled sandals": "Shoes",
    "Flat shoe": "Shoes",
    "Flip flop": "Shoes",
    "Slippers": "Shoes",
    # --- [액세서리 및 잡화] ---
    "Hat/beanie": "Accessories",
    "Cap/peaked": "Accessories",
    "Scarf": "Accessories",
    "Gloves": "Accessories",
    "Belt": "Accessories",
    "Sunglasses": "Accessories",
    "Bag": "Accessories",
    "Wallet": "Accessories",
    "Ring": "Accessories",
    "Earring": "Accessories",
    "Necklace": "Accessories",
    "Bracelet": "Accessories",
    "Hair/alice band": "Accessories",
    "Hair clip": "Accessories",
    "Other accessories": "Accessories",
    "Costumes": "Accessories",
    "Unknown": "Unknown",  # 끝까지 미분류인 경우
}

In [ ]:
EXCLUDED_PRODUCT_GROUPS = [
    "Items",
    "Furniture",
    "Garment and Shoe care",
    "Stationery",
    "Interior textile",
    "Fun",
    "Cosmetic",
    "Unknown",
]

NON_FASHION_TYPES = [
    "Washing bag",
    "Sewing kit",
    "Stain remover spray",
    "Clothing mist",
    "Wood balls",
    "Zipper head",
    "Marker pen",
    "Side table",
    "Soft Toys",
    "Waterbottle",
    "Giftbox",
    "Dog Wear",
    "Umbrella",
    "Sleeping sack",
    "Sleep Bag",
    "Baby Bib",
    "Nipple covers",
    "Bra extender",
]

### articles

1. detail_desc 결측치 416개 > prod_name으로 대체
2. 이미지 누락 442개 (약 0.42%) > 특정 카테고리 집중이 아닌 전반적으로 분산된 누락이라 제거
   - 모델 간 공정한 비교를 위해 모든 모델에서 동일하게 제거
3. product_type_name
   - 비의류 제거 (product_group_name 제거만으로 걸러지지 않는 항목 존재)
     > Washing bag, Sewing kit, Stain remover spray, Clothing mist, Wood balls,
       Zipper head, Marker pen, Side table, Soft Toys, Waterbottle, Giftbox,
       Dog Wear, Umbrella, Sleeping sack, Sleep Bag, Baby Bib,
       Nipple covers, Bra extender
     > 기준: 사람이 착용하는 의류/패션 아이템인가
   - 표기 불일치/유사 type 통합 (type_merge_mapping)
     > Earrings→Earring, Flat shoes→Flat shoe, Cap→Cap/peaked,
       헤어밴드류→Hair/alice band, 가방류→Bag, 모자류→Hat/beanie
   - 통합 후에도 50개 미만인 희귀 패션 type
     > 상품은 유지, type만 "Other_{product_group_name}"으로 병합
     > 이유: 신발 등 정상 패션 아이템 포함, 제거 시 정보 손실
   - 처리 순서: 비의류 제거 > 통합 > 빈도 재계산 > Other_ 병합
4. product_group_name
   - 비의류 카테고리 제거: Items, Furniture, Garment and Shoe care, Stationery,
     Interior textile, Fun, Cosmetic
   - Unknown 121건 (상품 0.1%, 거래 0.3%) > 제거
5. garment_group_name Unknown 3,846건 (3.6%)
   - 메타데이터 기반 유사도 계산 시 Unknown끼리 거짓 유사도를 만들 수 있어 라벨링 필요
   - product_type_name(68종) 기준으로 대표 이미지 확인 후 수작업 매핑
   - Bodysuit는 판단 애매하나 영향 미미하여 Jersey Basic으로 유지
6. 숨은 결측치: isnull()로 잡히지 않는 -1/Unknown 존재
   - 숫자 코드(-1)와 문자열(Unknown)이 같은 행에서 쌍으로 존재함을 확인
   - graphical_appearance_name(52), colour_group_name(28),
     perceived_colour_value_name(28), perceived_colour_master_name(685)
     > 비율이 작아 Unknown을 유효 카테고리로 유지
7. 계층 구조: index_group_name(5) > index_name(10) > section_name(56) > department_name(250)
   - 사용할 계층 수준은 피처 엔지니어링 단계에서 실험으로 결정

### 전처리 파이프라인 주의사항
- 처리 순서: 비의류/저빈도/이미지 누락 제거 > garment_group 매핑 > 결측 대체
- articles에서 제거된 상품은 transactions에서도 제거
- 제거 전후 상품 수, 거래 수 기록

In [ ]:
import numpy as np
import pandas as pd

transactions_df["t_dat"] = pd.to_datetime(transactions_df["t_dat"])

print(f"시작일: {transactions_df['t_dat'].min()}")
print(f"종료일: {transactions_df['t_dat'].max()}")
print(
    f"전체 기간: {(transactions_df['t_dat'].max() - transactions_df['t_dat'].min()).days}일"
)

In [ ]:
print(transactions_df.shape)
print(transactions_df.dtypes)
print(transactions_df.isnull().sum())

# 숨은 결측 후보: article_id, customer_id, price, sales_channel_id
for col in ["article_id", "price", "sales_channel_id"]:
    n_zero = (transactions_df[col] == 0).sum()
    n_neg = (
        (transactions_df[col] < 0).sum()
        if transactions_df[col].dtype != "object"
        else 0
    )
    print(f"{col}: 0값 {n_zero}건, 음수값 {n_neg}건")

In [ ]:
print(transactions_df["price"].describe())

zero_price = transactions_df[transactions_df["price"] == 0]
print(f"0원 거래: {len(zero_price)}건 ({len(zero_price) / len(transactions_df):.4%})")

# 상/하위 1% 확인
print(transactions_df["price"].quantile([0.001, 0.01, 0.99, 0.999]))

In [ ]:
user_purchase_counts = transactions_df.groupby("customer_id").size()

print(user_purchase_counts.describe())
print(f"1회 구매 유저 비율: {(user_purchase_counts == 1).mean():.4%}")
print(f"5회 이하 구매 유저 비율: {(user_purchase_counts <= 5).mean():.4%}")

In [ ]:
item_sales_counts = transactions_df.groupby("article_id").size()

print(item_sales_counts.describe())
print(f"1회만 팔린 상품 비율: {(item_sales_counts == 1).mean():.4%}")

# long-tail 시각화용 상위 1% 상품이 차지하는 거래 비중
top1pct_threshold = item_sales_counts.quantile(0.99)
top1pct_share = (
    item_sales_counts[item_sales_counts >= top1pct_threshold].sum()
    / item_sales_counts.sum()
)
print(f"상위 1% 상품이 전체 거래에서 차지하는 비중: {top1pct_share:.4%}")

In [ ]:
print(transactions_df["sales_channel_id"].value_counts(normalize=True))


In [ ]:
dup_counts = transactions_df.groupby(["customer_id", "article_id"]).size()

print(f"중복 구매(같은 유저-상품 2회 이상) 비율: {(dup_counts > 1).mean():.4%}")
print(dup_counts[dup_counts > 1].describe())

In [ ]:
# articles_df는 원본(전처리 전) 기준
excluded_ids = articles_df.loc[
    articles_df["product_group_name"].isin(EXCLUDED_PRODUCT_GROUPS)
    | articles_df["product_type_name"].isin(NON_FASHION_TYPES),
    "article_id",
]

excluded_txn = transactions_df["article_id"].isin(excluded_ids)
print(f"제거 대상 상품의 거래 비중: {excluded_txn.mean():.4%}")

In [ ]:
import matplotlib.pyplot as plt

max_date = transactions_df["t_dat"].max()

# 0 = 마지막 7일, 1 = 그 이전 7일, ...
transactions_df["week_idx"] = ((max_date - transactions_df["t_dat"]).dt.days // 7).astype("int16")

weekly_stats = transactions_df.groupby("week_idx").agg(
    n_transactions=("article_id", "size"),
    n_users=("customer_id", "nunique"),
    n_items=("article_id", "nunique"),
).sort_index(ascending=False)

print(weekly_stats.head(8).to_string())   # 가장 오래된 주
print(weekly_stats.tail(12).to_string())  # 최근 12주

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(-weekly_stats.index, weekly_stats["n_transactions"])
ax.set_xlabel("weeks before end (0 = last week)")
ax.set_ylabel("transactions")
ax.set_title("Weekly transactions")
plt.show()

In [ ]:
weekly_dates = transactions_df.groupby("week_idx").agg(
    start=("t_dat", "min"),
    end=("t_dat", "max"),
    n_transactions=("article_id", "size"),
    online_ratio=("sales_channel_id", lambda s: (s == 2).mean()),
)

print(weekly_dates.nlargest(6, "n_transactions").to_string())
print(weekly_dates.loc[20:30].to_string())

In [ ]:
for test_weeks in [1, 2, 4]:
    is_test = transactions_df["week_idx"] < test_weeks
    train_part = transactions_df.loc[~is_test]
    test_part = transactions_df.loc[is_test]

    train_items = train_part["article_id"].unique()
    train_users = train_part["customer_id"].unique()

    new_item_txn = ~test_part["article_id"].isin(train_items)
    new_user_txn = ~test_part["customer_id"].isin(train_users)

    print(f"[test = 마지막 {test_weeks}주] 거래 {len(test_part):,}건")
    print(f"  신규 상품 거래 비중: {new_item_txn.mean():.2%} "
          f"(상품 수 {test_part.loc[new_item_txn, 'article_id'].nunique():,}개)")
    print(f"  신규 유저 거래 비중: {new_user_txn.mean():.2%}")

# transactions_df EDA 정리

데이터: 2018-09-20 ~ 2020-09-22 (733일), 31,788,324건

---

## 1. 결측치 확인

확인 이유
- articles에서 isnull()은 0인데 실제로는 -1, Unknown으로 결측이 채워져 있었다.
  transactions도 같은 방식일 수 있어서 NaN뿐 아니라 0, 음수까지 확인했다.

결과
- NaN, 0, 음수 모두 없음.

진행 방향
- transactions는 결측 처리 없이 그대로 사용한다.

---

## 2. 가격 분포

확인 이유
- 0원 거래는 무료 증정이나 기록 오류일 가능성이 있어서, 실제 구매 의사가 담긴 거래인지 봐야 했다.
- 가격이 극단적으로 튀는 값이 있으면 나중에 가격을 feature로 쓸 때 스케일이 망가진다.

결과
- 0원 거래 없음.
- 평균 0.0278 > 중앙값 0.0254, 오른쪽 꼬리가 있는 분포.
- 99.9% 지점 0.168, 최댓값 0.59. 꼬리는 있지만 극단적이진 않다.

진행 방향
- 가격 기준으로 제거하거나 클리핑하지 않는다.
- 가격은 CF 단계에선 쓰지 않고, 이후 feature로 쓸 때 스케일링 여부를 다시 본다.

---

## 3. 유저당 구매 횟수

확인 이유
- CF는 유저의 구매 기록으로 유저 벡터를 학습한다. 구매가 적은 유저는 벡터가 부정확해서
  추천 품질이 떨어진다(cold-start). 이런 유저가 얼마나 되는지 알아야 평가 설계를 할 수 있다.

결과
- 중앙값 9회. 1회 구매 유저 9.65%, 5회 이하 36.8%.

진행 방향
- 유저의 1/3 이상이 구매 기록이 적다.
- 전체 성능만 보면 이 차이가 가려지므로 구매 횟수 기준으로 유저 그룹을 나눠 성능을 따로 본다.

---

## 4. 상품당 판매 횟수

확인 이유
- 상품 벡터도 판매 기록으로 학습된다. 많이 팔린 상품은 벡터가 정확하고,
  적게 팔린 상품은 벡터가 거의 노이즈다.
- 이미지 임베딩은 판매 횟수와 상관없이 모든 상품에 만들 수 있다.
  그래서 판매가 적은 상품이 많을수록 이미지 feature가 쓸모 있을 여지가 크다.

결과
- 중앙값 65회, 1회만 팔린 상품 4.3%.
- 상위 1% 상품이 전체 거래의 18.6%를 차지. 인기 상품 쏠림이 강한 롱테일 분포.

진행 방향
- 가설: 이미지 feature는 인기 상품보다 판매가 적은 상품에서 효과가 클 것이다.
- 상품을 판매 빈도 구간으로 나눠 구간별 성능을 비교해서 이 가설을 검증한다.

---

## 5. 같은 상품 중복 구매

확인 이유
- 구매 기록을 "샀다/안 샀다"로만 볼지, "몇 번 샀는지"까지 쓸지 정해야 한다.
  재구매가 드물면 횟수 정보는 의미가 없다.

결과
- 같은 유저가 같은 상품을 2번 이상 산 경우 12.87%, 대부분 2회.

진행 방향
- ALS에서 구매 횟수를 신뢰도 가중치(1 + α × 횟수)로 활용한다.
- BPR은 구조상 횟수를 쓰지 않으므로 두 모델 비교 시 이 차이를 같이 기록한다.

---

## 6. 판매 채널 (sales_channel_id)

확인 이유
- 서비스는 온라인 추천인데 데이터엔 매장 구매도 섞여 있다.
  합쳐서 쓸지 온라인만 쓸지 정하려면 어떤 값이 온라인인지부터 알아야 했다.

결과
- 채널 2 70.4%, 채널 1 29.6%.
- 2020-03-25 ~ 05-05에 채널 2 비율 100%. 코로나 매장 폐쇄 시기와 겹친다.
  → 채널 2 = 온라인으로 판단. (공식 문서에 나온 건 아니고 데이터로 추론한 것)

진행 방향
- 매장 구매도 그 사람의 취향을 반영하므로 두 채널을 합쳐서 쓴다.
- 온라인만 빼면 거래 30%가 사라지고 구매 기록이 적은 유저가 더 늘어난다.
- 온라인만 쓰는 버전은 나중에 비교 실험으로 남겨둔다.

---

## 7. 주별 거래량 추이

확인 이유
- 시간 순서로 train/valid/test를 나눌 때, 평가 구간에 세일 같은 특이한 주가 들어가면
  평소와 다른 구매 패턴으로 모델을 평가하게 된다. 어느 구간이 안정적인지 봐야 했다.

결과
- 6월 말(여름 세일), 11월 말(블랙프라이데이)에 거래량이 급증하고 매년 반복된다.
- 최근 5주는 급증/급감 없이 25~30만 건 수준으로 안정적.
- 주 단위는 종료일부터 7일씩 거꾸로 잘랐다(week_idx, 0 = 마지막 주).
  달력 기준으로 자르면 마지막 주가 7일이 안 되기 때문.

진행 방향
- test: week 0 (마지막 7일)
- valid: week 1 (test 직전 7일)
- train: week 2 이전 전체
- 인기 상품 baseline은 전체 기간 버전과 최근 1주 버전을 둘 다 만든다.
  전체 기간으로 세면 세일 때 몰려 팔린 상품이 상위에 올라오기 때문.

---

## 8. test 기간의 신규 상품 / 신규 유저

확인 이유
- train에 한 번도 없던 상품·유저는 CF가 벡터를 만들 수 없어서 아예 추천을 못 한다.
  test 기간을 얼마나 잡느냐에 따라 이 비율이 달라지므로 길이를 정하는 근거로 봤다.

결과
| test 길이 | 신규 상품 거래 비중 | 신규 유저 거래 비중 |
|---|---|---|
| 1주 | 3.98% (667개) | 7.55% |
| 2주 | 9.07% (1,580개) | 7.56% |
| 4주 | 15.64% (3,396개) | 7.65% |

- test 기간이 길수록 신규 상품 비중이 빠르게 커진다. 신규 유저 비중은 거의 그대로.

진행 방향
- test를 1주로 잡는다. 기간이 길면 모델 실력보다 신규 상품이 얼마나 섞였는지가 결과를 좌우한다.
  원 대회도 다음 7일 예측이라 비교하기 쉽다.
- 신규 상품 구간은 CF가 못 맞히는 부분이라 이미지 feature 효과를 따로 떼서 본다.
- 신규 유저는 인기 상품 추천으로 채우고, 모든 모델에 같은 방식을 적용해 비교를 공정하게 한다.

---

## 9. articles 전처리로 제거될 상품의 거래 비중

확인 이유
- articles에서 비의류, Unknown 등을 지우기로 했는데, 이 상품들이 거래에서 큰 비중이면
  정제가 아니라 데이터 손실이 된다.

결과
- 전체 거래의 0.39%.

진행 방향
- 영향이 작아서 계획대로 제거하고, transactions에서도 같은 article_id를 제거한다.